---
### load imports and set up spark session
---

In [13]:
import sys
import os
from pathlib import Path
import uuid

sys.path.insert(0, str(Path("../").resolve()))
from Configs.bronze_schemas import SCHEMAS

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

from delta.tables import DeltaTable
from delta import configure_spark_with_delta_pip

warehouse_path = Path.home() / ".local/share/Project_Mercury/spark-warehouse"

# trust_store = Path("/mnt/c/Users/313983/AppData/Local/Project_Mercury/cacerts")
# os.environ["JAVA_TOOL_OPTIONS"] = (
#     f"-Djavax.net.ssl.trustStore={trust_store} "
#     "-Djavax.net.ssl.trustStorePassword=changeit"
# )

builder = (
    SparkSession.builder
    .appName("data_pipeline")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config("spark.sql.warehouse.dir", warehouse_path.as_uri())
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/10/01 19:54:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [14]:
from IPython.core.magic import register_cell_magic


@register_cell_magic
def sql(line, cell):
    spark.sql(cell).show()

In [15]:

RUN_ID = str(uuid.uuid4())
BATCH_ID = "B001" # toggle to "B002"
SOURCE_SYSTEM = "OLTP"

ENTITIES = [
    "address",
    "category",
    "customer",
    "order",
    "order_item",
    "payment",
    "product",
    "product_category",
    "return",
    "return_item"
]

In [16]:
def resolve_table_path(table_name):
    return Path(f"../data/{BATCH_ID}/{table_name}_{BATCH_ID}.csv").resolve()


def resolve_entity_schema(entity):
    schema = SCHEMAS[entity]

    if "_corrupt_record" in schema.fieldNames():
        return schema

    return StructType(
        schema.fields + [
            StructField("_corrupt_record", StringType(), True)
        ]
    )

In [17]:
def read_table(schema, path):
    df = (
        spark.read
        .option("header", "true")
        .schema(schema)
        .option("delimiter", ",")
        .option("multiline", "true")
        .option("mode", "permissive")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .csv(str(path))
    )
    return df

In [18]:
def load_df(entity):
    table_schema = resolve_entity_schema(entity)
    table_path = resolve_table_path(entity)
    df = read_table(table_schema, table_path)
    return df

In [19]:
def add_metadata_columns(df):
    return (df
        .withColumn("_record_hash", F.sha2(F.concat_ws("||", *df.columns), 256))
        .withColumn("_batch_id", F.lit(BATCH_ID))
        .withColumn("_run_id", F.lit(RUN_ID))
        .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("_ingested_timestamp", F.current_timestamp())
    )

#### idempotent write

- if match on source_record and hash don't insert
- if match on source_record and not on hash throw error
- if no match insert new row

In [ ]:
def idempotent_write(df, target):
    table_name = f"bronze.{target}"
    table_path = warehouse_path / "bronze.db" / target

    if not spark.catalog.tableExists(table_name):
        if DeltaTable.isDeltaTable(spark, str(table_path)):
            quoted_table_name = f"`bronze`.`{target.replace('`', '``')}`"
            location = table_path.as_uri().replace("'", "''")
            spark.sql(
                f"CREATE TABLE {quoted_table_name} USING DELTA "
                f"LOCATION '{location}'"
            )
        elif table_path.exists():
            raise RuntimeError(
                f"Cannot register {table_name}: location exists but is not "
                f"a Delta table: {table_path}"
            )
        else:
            df.write.format("delta").saveAsTable(table_name)
            return

    target_df = spark.table(table_name)
    key_condition = (
        (F.col("s._source_system") == F.col("t._source_system"))
        & (F.col("s.source_record_id") == F.col("t.source_record_id"))
    )

    conflicting_record = (
        df.alias("s")
        .join(target_df.alias("t"), key_condition, "inner")
        .where(~F.col("s._record_hash").eqNullSafe(F.col("t._record_hash")))
        .select("s._source_system", "s.source_record_id")
        .limit(1)
        .first()
    )

    if conflicting_record is not None:
        raise ValueError(
            f"Conflicting content for {table_name}: source_system="
            f"{conflicting_record['_source_system']}, "
            f"source_record_id={conflicting_record['source_record_id']}"
        )

    match_condition = (
        "t._source_system = s._source_system "
        "AND t.source_record_id = s.source_record_id"
    )
    (
        DeltaTable.forName(spark, table_name)
        .alias("t")
        .merge(df.alias("s"), match_condition)
        .whenNotMatchedInsertAll()
        .execute()
    )

In [21]:
df_dict = {}
for entity in ENTITIES:
    df = load_df(entity)
    df_with_metadata = add_metadata_columns(df)
    df_dict[entity] = df_with_metadata
    idempotent_write(df_with_metadata, entity)
    df_dict[entity].show(5, truncate=False)

+--------------------------+----------+-----------+------------+---------------+--------------+----------+---------------+-----------+------------+----------+---------------------------+---------------------------+---------------+----------------------------------------------------------------+---------+------------------------------------+--------------+--------------------------+
|source_record_id          |address_id|customer_id|address_type|address_line_1 |address_line_2|city      |state_region   |postal_code|country_code|is_primary|created_at                 |updated_at                 |_corrupt_record|_record_hash                                                    |_batch_id|_run_id                             |_source_system|_ingested_timestamp       |
+--------------------------+----------+-----------+------------+---------------+--------------+----------+---------------+-----------+------------+----------+---------------------------+---------------------------+---------------+

+-----------------------------+-------------+----------+----------+--------+----------+---------------+----------+-----------------+---------------------------+---------------------------+---------------+----------------------------------------------------------------+---------+------------------------------------+--------------+--------------------------+
|source_record_id             |order_item_id|order_id  |product_id|quantity|unit_price|discount_amount|tax_amount|line_total_amount|created_at                 |updated_at                 |_corrupt_record|_record_hash                                                    |_batch_id|_run_id                             |_source_system|_ingested_timestamp       |
+-----------------------------+-------------+----------+----------+--------+----------+---------------+----------+-----------------+---------------------------+---------------------------+---------------+----------------------------------------------------------------+---------+---